# EDA: `raw_airline_reviews`

One notebook per raw Skytrax table (index in `eda.ipynb`).

**Role of this table:** Fact table of airline (flight) reviews, one row per review. The largest table. Joins to `raw_airlines` on `airline_id`.

**Read-only:** the CSV in `data/` is only read. Cleaning below happens on an in-memory copy (`df`); the real cleaning rules are written later in dbt staging.

## Table of Contents

- [0. Setup](#sec-0)
  - [0.1 Imports](#sec-0-1)
  - [0.2 Config](#sec-0-2)
  - [0.3 Plot style](#sec-0-3)
  - [0.4 Helpers](#sec-0-4)
- [1. Look at the Big Picture](#sec-1)
  - [1.1 Frame the Problem](#sec-1-1)
  - [1.2 Questions & Hypotheses](#sec-1-2)
  - [1.3 Performance Measure (for later modelling)](#sec-1-3)
  - [1.4 Check the Assumptions](#sec-1-4)
- [2. Get the Data](#sec-2)
  - [2.1 Data Dictionary](#sec-2-1)
- [3. Data Cleaning (General)](#sec-3)
  - [3.1 Format Column Names](#sec-3-1)
  - [3.2 Format Data Body](#sec-3-2)
  - [3.3 Format Datatypes](#sec-3-3)
  - [3.4 Duplicates](#sec-3-4)
  - [3.5 Domain Sanity Checks](#sec-3-5)
- [4. Data Quality](#sec-4)
  - [4.1 Missing Values](#sec-4-1)
  - [4.2 Outliers](#sec-4-2)
- [5. Attribute Combination](#sec-5)
- [6. EDA](#sec-6)
  - [6.0 Classify Attributes](#sec-6-0)
  - [6.1 Univariate](#sec-6-1)
  - [6.2 Bivariate](#sec-6-2)
  - [6.3 Multivariate](#sec-6-3)
- [7. Summary of Findings](#sec-7)
- [8. Hand-off to Modelling](#sec-8)

# 0. Setup<a id="sec-0"></a>

## 0.1 Imports<a id="sec-0-1"></a>

In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

## 0.2 Config<a id="sec-0-2"></a>

Everything dataset-specific lives here. Column names are the **snake_case** names produced in section 3.1.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TABLE = "airline_reviews"            # name of the file in data/
DATA_PATH = ROOT / "data" / f"{TABLE}.csv"
TARGET = "recommended"        # yes/no: the business question is what goes with a recommendation
PROBLEM_TYPE = "classification"   # nothing is predicted: this only switches on the target views of the template
ID_COLS = ['review_id']       # identifiers: excluded from analysis
CAT_COLS = ['airline_id']     # force these to categorical; None = auto-detect
CAT_MAX_UNIQUE = 15           # numeric columns with <= this many values are treated as discrete
HIGH_CORR = 0.8               # |r| above this is flagged as multicollinearity
TOP_N_CATEGORIES = 10         # categories shown per bar chart

SAVE_FIGS = False             # True writes every figure to FIG_DIR
FIG_DIR = ROOT / "reports" / "figures"
RANDOM_STATE = 42

## 0.3 Plot style<a id="sec-0-3"></a>

One consistent look: blue for single-series charts, a fixed categorical order for
hue splits (max 3 groups per chart), and a blue ↔ gray ↔ red diverging map for correlations.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MAIN = PALETTE[0]
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#86b6ef", "#f0efec", "#f19a99", "#b52f2e"])

sns.set_theme(
    style="whitegrid",
    palette=PALETTE,
    rc={
        "axes.edgecolor": "#c3c2b7",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelcolor": "#52514e",
        "grid.color": "#ecebe7",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "lines.linewidth": 2,
        "figure.dpi": 100,
    },
)

## 0.4 Helpers<a id="sec-0-4"></a>

In [ ]:
def to_snake_case(df):
    """CamelCase / spaces / dashes -> snake_case column names."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.replace(r"(?<=[a-z0-9])(?=[A-Z])", "_", regex=True)
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.lower()
    )
    return df


def save_fig(fig, name):
    if SAVE_FIGS and name:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=110, bbox_inches="tight")


def plot_grid(columns, plot_fn, ncols=4, height=3.0, title=None, name=None, shared_legend=False):
    """Draw plot_fn(col, ax) for every column in a grid of subplots.

    shared_legend=True replaces per-panel legends with one legend above the grid.
    """
    columns = list(columns)
    if not columns:
        print("Nothing to plot.")
        return
    nrows = math.ceil(len(columns) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, height * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, col in zip(axes, columns):
        plot_fn(col, ax)
        ax.set_title(col)
    for ax in axes[len(columns):]:
        ax.axis("off")
    fig.tight_layout()
    legend = axes[0].get_legend()
    if shared_legend and legend:
        handles, labels = legend.legend_handles, [t.get_text() for t in legend.get_texts()]
        fig.legend(handles, labels, title=legend.get_title().get_text(), loc="lower center",
                   bbox_to_anchor=(0.5, 1.0), ncol=len(labels), frameon=False)
        for ax in axes:
            if ax.get_legend():
                ax.get_legend().remove()
    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.0 + (0.7 / fig.get_figheight() if shared_legend and legend else 0))
    save_fig(fig, name)
    plt.show()


def missing_report(df):
    """Count and % of nulls per column (only columns with nulls)."""
    report = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean() * 100,
        "dtype": df.dtypes.astype(str),
    })
    return report[report.n_missing > 0].sort_values("n_missing", ascending=False)


def outlier_report(df, columns, k=1.5):
    """IQR-based outlier counts. Detection only - nothing is removed."""
    rows = []
    for col in columns:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        n_out = int(((s < lo) | (s > hi)).sum())
        rows.append({"column": col, "lower": lo, "upper": hi, "min": s.min(), "max": s.max(),
                     "n_outliers": n_out, "pct_outliers": 100 * n_out / len(s)})
    return pd.DataFrame(rows).set_index("column").sort_values("n_outliers", ascending=False)


def classify_columns(df, target=None, id_cols=(), cat_cols=None, cat_max_unique=15):
    """Split columns into constant / binary / categorical / discrete / continuous."""
    groups = {"constant": [], "binary": [], "categorical": [], "discrete": [], "continuous": []}
    forced_cat = set(cat_cols or [])
    for col in df.columns:
        if col in id_cols or col == target:
            continue
        n = df[col].nunique(dropna=True)
        if n <= 1:
            groups["constant"].append(col)
        elif n == 2:
            groups["binary"].append(col)
        elif col in forced_cat or not pd.api.types.is_numeric_dtype(df[col]):
            groups["categorical"].append(col)
        elif n <= cat_max_unique:
            groups["discrete"].append(col)
        else:
            groups["continuous"].append(col)
    return groups

# 1. Look at the Big Picture<a id="sec-1"></a>

## 1.1 Frame the Problem<a id="sec-1-1"></a>

Answered before touching the data. Business context: the stakeholder is the Head of Customer Experience of a mid-size full-service airline, who decides how to split the yearly budget for improving the customer experience.

* **What is the business objective?**
  Decide where to spend the yearly budget for improving the customer experience (seat, food, wifi, entertainment, cabin crew, ground service) across economy and premium cabins.
* **How will the output be used?**
  The Head of Customer Experience of a mid-size full-service airline uses it to choose which service aspect to fund first, and to see where the airline trails its peers.
* **What does the target look like?** (regression / classification / none)
  `recommended` (yes/no), so a classification-style view. Nothing is predicted: we only look at which aspects go together with a customer recommending the airline.
* **What would a useful result be?**
  A ranked list of aspects by how strongly they go with `recommended`, per cabin, plus where scores are lowest, with clear data limits.

Main question (all review tables): *which service aspect goes together most with customers recommending the airline, and where is the airline weaker than its peers?*

## 1.2 Questions & Hypotheses<a id="sec-1-2"></a>

The questions this EDA must answer. Each one maps to a section below. Q-numbers follow the business question list; `D1` is the data-readiness check that comes first.

| # | Question | Answered in |
|---|---|---|
| D1 | Can we trust the data enough to answer (nulls, duplicates, typos, wrong dates)? | 3, 4 |
| Q1 | In each cabin, which aspects score lowest, and how many airlines have at least 100 reviews (enough to be a peer)? | 6.1, 6.3.1 |
| Q2 | Which aspect goes together most with `recommended`, and does this change by cabin? | 6.2 |
| Q3 | How do scores and the recommend rate move over time (separating the whole industry from one airline)? | 6.3.1 |
| Q4 | What is the recommend rate by cabin and by traveller type? | 6.3.1 |
| Q5 | Does `value_for_money` move with the other aspects (is spending on one aspect risky for value)? | 6.2, 6.3 |
| Q6 | Which traveller groups are least satisfied? | 6.2, 6.3.1 |
| Q7 | Are some ratings so correlated that they are redundant? | 6.3 |

## 1.3 Performance Measure (for later modelling)<a id="sec-1-3"></a>

Pick now so the EDA looks at the right things (e.g. skewed target → consider RMSLE / log target).

* **Regression:** RMSE, MAE, MAPE, RMSLE, adjusted R²
* **Classification:** ROC-AUC, PR-AUC, F1, precision/recall, confusion matrix (check class balance in 6.1)

*Here there is no model, so the "performance measure" is a business one: the recommend rate and the mean-score gap per aspect (built later in the marts).*

## 1.4 Check the Assumptions<a id="sec-1-4"></a>

To confirm with the data owner / stakeholder.

* `review_id` is the primary key; one row = one flight review.
* `date_flown` is the first day of the flight month, not the exact day.
* Airport columns are 3-letter IATA codes; city columns are typed by reviewers and are messy.
* The gap to peers needs airline attributes (service model, region, peer group) that are not in the data, so they come later from a seed.
* Reviews are voluntary opinions posted on Skytrax, not a random survey: results describe people who write reviews.
* `recommended` measures how appealing an airline is, not whether the ticket was bought. We say "goes with", never "causes".
* Scores differ strongly by region and nationality, so only compare like with like (same cabin, same region).

# 2. Get the Data<a id="sec-2"></a>

In [ ]:
raw = pd.read_csv(DATA_PATH)   # the CSV is only read, never changed
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")

In [ ]:
# Take a look at the top 5 rows
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

## 2.1 Data Dictionary<a id="sec-2-1"></a>

One row per column: what it means, its type, how many distinct values, and an example. The meanings are inferred from the column names and the data.

In [ ]:
# Meaning of each column (from the column name and the data; correct them if you know better)
MEANINGS = {
    "review_id": "Review id (primary key)",
    "verify": "Verified review",
    "date_submitted": "Date the review was posted",
    "customer_name": "Reviewer name (ends with an anonymisation number)",
    "nationality": "Reviewer country",
    "recommended": "Reviewer recommends it",
    "review": "Free-text review",
    "updated_at": "Load time",
    "airline_id": "Airline reviewed, joins to `raw_airlines`",
    "date_flown": "Month of the flight (first day of month)",
    "type_of_traveller": "Trip type",
    "seat_type": "Cabin class",
    "aircraft": "Aircraft model flown (free text)",
    "origin_city": "City where the flight started",
    "origin_airport": "Airport (3-letter code) where the flight started",
    "destination_city": "City where the flight ended",
    "destination_airport": "Airport (3-letter code) where the flight ended",
    "transit_city": "Stop-over city, if any",
    "transit_airport": "Stop-over airport, if any",
    "seat_comfort": "Rating 1 to 5",
    "cabin_staff_service": "Rating 1 to 5",
    "food_and_beverages": "Rating 1 to 5",
    "inflight_entertainment": "Rating 1 to 5",
    "ground_service": "Rating 1 to 5",
    "wifi_and_connectivity": "Rating 1 to 5",
    "value_for_money": "Rating 1 to 5",
}

data_dictionary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_unique": raw.nunique(),
    "example": raw.iloc[0],
    "meaning": pd.Series(MEANINGS),
}).reindex(raw.columns)
data_dictionary

# 3. Data Cleaning (General)<a id="sec-3"></a>

Cleaning that is safe to do on the **whole** dataset (no information from the target or test set is used).

## 3.1 Format Column Names<a id="sec-3-1"></a>

In [ ]:
df = to_snake_case(raw)
df.columns.tolist()

In [ ]:
# Manual renames, if any
# df = df.rename(columns={"unnamed:_0": "id"})

## 3.2 Format Data Body<a id="sec-3-2"></a>

Trim whitespace in text columns, then pull structured fields out of free-text columns.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())
for col in text_cols:
    print(f"{col}: {df[col].nunique()} unique -> {[str(v)[:40] for v in df[col].unique()[:8]]}")

`customer_name` ends with an anonymisation number (e.g. `Joshua Torres 6`), which is not part of a real name. Strip it so the same person is written the same way.

In [ ]:
has_suffix = df["customer_name"].str.contains(r"\s\d+$", na=False)
print(f"{has_suffix.sum():,} of {len(df):,} names end with a number")
df["customer_name"] = df["customer_name"].str.replace(r"\s+\d+$", "", regex=True)
pd.DataFrame({"before": raw["customer_name"], "after": df["customer_name"]}).loc[has_suffix].head(5)

## 3.3 Format Datatypes<a id="sec-3-3"></a>

Check every column has the right type: dates → `datetime64`, flags → `int`/`bool`, labels → `category`.

In [ ]:
df.dtypes.value_counts()

In [ ]:
# Dates must be datetime, flags must be nullable booleans (True / False / missing)
for col in [c for c in df.columns if c.startswith("date_") or c == "updated_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in [c for c in ["verify", "recommended"] if c in df.columns]:
    df[col] = df[col].astype("boolean")
df.dtypes.value_counts()

## 3.4 Duplicates<a id="sec-3-4"></a>

In [ ]:
analysis_cols = [c for c in df.columns if c not in ID_COLS]
n_exact = df.duplicated().sum()
n_ignoring_id = df.duplicated(subset=analysis_cols).sum()
print(f"Exact duplicate rows:            {n_exact}")
print(f"Duplicates ignoring ID columns:  {n_ignoring_id}")
df[df.duplicated(subset=analysis_cols, keep=False)]

In [ ]:
# Decide: drop or keep? Duplicates ignoring the ID usually mean the same record was loaded twice.
df = df.drop_duplicates(subset=analysis_cols).reset_index(drop=True)
print(f"{len(df):,} rows after de-duplication")

## 3.5 Domain Sanity Checks<a id="sec-3-5"></a>

Values that are *possible* to store but *impossible* in reality (placeholder dates, wrong-column values, leftovers).
These are not statistical outliers; they are data-entry or scraping errors. Wrong dates are set to missing here, like the staging model will do.

In [ ]:
# Dates: 1970-01-01 is a placeholder for "no date", not a real date -> set to missing
date_cols = [c for c in df.select_dtypes(include="datetime").columns if c != "updated_at"]
for col in date_cols:
    n_placeholder = (df[col] < "1990-01-01").sum()
    print(f"{col}: {n_placeholder} placeholder dates (before 1990)")
    df.loc[df[col] < "1990-01-01", col] = pd.NaT

# The trip date can never be later than the day the review was written
visit_col = next((c for c in ["date_flown", "date_visit"] if c in df.columns), None)
if visit_col:
    wrong_order = df[visit_col] > df["date_submitted"]
    print(f"{visit_col} later than date_submitted: {wrong_order.sum()} rows (set to missing)")
    display(df.loc[wrong_order, ["review_id", "date_submitted", visit_col]].head())
    df.loc[wrong_order, visit_col] = pd.NaT

# Ratings must be whole numbers from 1 to 5
rating_cols = df.select_dtypes("float").columns.tolist()
rating_problems = {
    c: int(((df[c].dropna() < 1) | (df[c].dropna() > 5) | (df[c].dropna() % 1 != 0)).sum())
    for c in rating_cols
}
print("Ratings outside 1-5 or not whole numbers:", {c: n for c, n in rating_problems.items() if n} or "none")

# Route: origin and destination should differ
same_city = (df["origin_city"].notna() & (df["origin_city"] == df["destination_city"])).sum()
same_airport = (df["origin_airport"].notna() & (df["origin_airport"] == df["destination_airport"])).sum()
print(f"origin = destination: {same_city} rows by city, {same_airport} rows by airport")

# Cities filled but airport code missing (airport columns are much emptier than city columns)
for prefix in ["origin", "destination", "transit"]:
    n = (df[f"{prefix}_city"].notna() & df[f"{prefix}_airport"].isna()).sum()
    print(f"{prefix}: {n:,} rows have a city but no airport code")

# Airport code typed in a city column (3 letters): the staging model moves it to the airport column
for prefix in ["origin", "destination", "transit"]:
    n = df[f"{prefix}_city"].str.fullmatch(r"[A-Za-z]{3}", na=False).sum()
    print(f"{prefix}_city holds a 3-letter code in {n:,} rows")

# nationality should be a country; digits, dots, or very short / very long values are leaked text
nat = df["nationality"].dropna()
bad_nat = nat.str.contains(r"\d|\.") | (nat.str.len() <= 2) | (nat.str.len() > 25)
print(f"nationality: {bad_nat.sum()} of {len(nat):,} values do not look like a country")
nat[bad_nat].value_counts().head(10).to_frame()

# Very short reviews: often only a marker such as "Trip Verified" with no real text
short_review = df[df["review"].str.len() < 20]
print(f"{len(short_review)} reviews shorter than 20 characters")
short_review[["review_id", "review"]].head()

# Same review text under different review_id (exact duplicates were already dropped in 3.4)
has_text = df["review"].notna()
same_text = df[has_text].duplicated(subset=["review"], keep=False)
print(f"{same_text.sum():,} rows share their review text with another row")
same_text_airline = df[has_text].duplicated(subset=["airline_id", "review"], keep=False)
print(f"{same_text_airline.sum():,} of them are the same airline with the same text (true duplicates)")


# 4. Data Quality<a id="sec-4"></a>

## 4.1 Missing Values<a id="sec-4-1"></a>

**Types of nulls** (decides how to treat them later):
* **MCAR** - missing completely at random (lost in ETL). Drop or impute.
* **MAR** - missingness explained by another observed column. Impute using that column.
* **MNAR** - missingness depends on the missing value itself (e.g. high earners hiding income). Add a "was missing" flag; model carefully.
* **By design** - field not applicable (e.g. no cancellation date for active customers). Fill with a sentinel.

**Options:** drop rows · drop column · impute (median / mode / model-based) · add an `is_missing` indicator.
*Only decide here; fit imputers on the training set after the split.*

In [ ]:
missing = missing_report(df)
print(f"{df.isna().sum().sum()} missing values in {len(missing)} columns; "
      f"{df.isna().any(axis=1).sum()} rows ({df.isna().any(axis=1).mean():.1%}) affected")
missing

In [ ]:
if len(missing):
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(missing) + 1))
    ax.barh(missing.index, missing["pct_missing"], color=MAIN, height=0.6)
    ax.invert_yaxis()
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column")
    for i, v in enumerate(missing["pct_missing"]):
        ax.text(v, i, f" {v:.1f}%", va="center", color="#52514e", fontsize=9)
    save_fig(fig, "missing_values")
    plt.show()

In [ ]:
# Do nulls co-occur? (same rows missing several fields -> one upstream cause)
if len(missing) > 1:
    display(df[missing.index].isna().corr().round(2))

In [ ]:
# Rows with a missing TARGET cannot be used for supervised training
if TARGET and df[TARGET].isna().any():
    print(f"{df[TARGET].isna().sum()} rows have no {TARGET} -> drop before modelling")

## 4.2 Outliers<a id="sec-4-2"></a>

**Detect, don't delete.** Flag values outside `Q1 - 1.5·IQR` / `Q3 + 1.5·IQR`, then decide per column:
real but rare (keep, maybe log-transform) vs. error (fix in 3.5).

In [ ]:
num_cols = [c for c in df.select_dtypes("number").columns
            if c not in ID_COLS and df[c].nunique() > 2]
if num_cols:
    outliers = outlier_report(df, num_cols)
    display(outliers[outliers.n_outliers > 0])
else:
    print("No numeric columns besides the id: nothing to check.")

In [ ]:
plot_grid(num_cols, lambda c, ax: sns.boxplot(x=df[c], ax=ax, color=MAIN, fliersize=3, linewidth=1),
          title="Box plots (numeric columns)", name="boxplots")

# 5. Attribute Combination<a id="sec-5"></a>

Create features that may explain the data better than the raw columns (ratios, rates, flags, date parts).
They go through the same EDA as everything else below.

how long after the trip the review was written, review length, how many ratings were filled in, review year, and route flags.

In [ ]:
# How long after the trip was the review written, how long is it, and how many ratings were filled in?
visit_col = next((c for c in ["date_flown", "date_visit"] if c in df.columns), None)
rating_cols = df.select_dtypes("float").columns.tolist()
if visit_col:
    df["days_to_review"] = (df["date_submitted"] - df[visit_col]).dt.days
df["review_length"] = df["review"].str.len()
df["n_ratings_filled"] = df[rating_cols].notna().sum(axis=1)
df["submitted_year"] = df["date_submitted"].dt.year

# Route flags
df["has_transit"] = df["transit_city"].notna() | df["transit_airport"].notna()
df["has_origin_airport_code"] = df["origin_airport"].notna()


# 6. EDA<a id="sec-6"></a>

## 6.0 Classify Attributes<a id="sec-6-0"></a>

Every plot below is driven by these groups. Override with `CAT_COLS` in the config if a group looks wrong.

In [ ]:
groups = classify_columns(df, TARGET, ID_COLS, CAT_COLS, CAT_MAX_UNIQUE)
for name, cols in groups.items():
    print(f"{name:<12} ({len(cols):>2}): {cols}")

const_attribs = groups["constant"]
bin_attribs = groups["binary"]
cat_attribs = groups["categorical"]
disc_attribs = groups["discrete"]
cont_attribs = groups["continuous"]
num_attribs = disc_attribs + cont_attribs

In [ ]:
# Constant columns carry no information
if const_attribs:
    print("Dropping constant columns:", const_attribs)
    df = df.drop(columns=const_attribs)

# High-cardinality text columns (e.g. free text, raw model names) swamp plots
high_card = [c for c in cat_attribs if df[c].nunique() > 50]
print("High-cardinality categoricals (summarised, not plotted):", high_card)
cat_plot_attribs = [c for c in cat_attribs if c not in high_card]

## 6.1 Univariate<a id="sec-6-1"></a>

One column at a time: shape, spread, skew, rare levels.

### 6.1.1 Target

The target is `recommended`: the share of reviews that recommend (class balance). A rate far from 50% is normal here; it is the baseline every segment is compared with.

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    y = df[TARGET].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(y, kde=True, ax=axes[0], color=MAIN)
    axes[0].set_title(f"{TARGET} (skew = {y.skew():.2f})")
    sns.histplot(np.log1p(y), kde=True, ax=axes[1], color=MAIN)
    axes[1].set_title(f"log1p({TARGET}) (skew = {np.log1p(y).skew():.2f})")
    fig.tight_layout()
    save_fig(fig, "target_distribution")
    plt.show()
    display(y.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame())
elif TARGET:
    counts = df[TARGET].value_counts()
    display(pd.DataFrame({"count": counts, "pct": counts / counts.sum() * 100}))
    fig, ax = plt.subplots(figsize=(6, 3))
    counts.plot.bar(ax=ax, color=MAIN)
    ax.set_title(f"Class balance: {TARGET}")
    save_fig(fig, "target_distribution")
    plt.show()

### 6.1.2 Numeric

In [ ]:
skew = df[num_attribs].skew().sort_values(key=abs, ascending=False).to_frame("skew")
skew["shape"] = pd.cut(skew["skew"].abs(), [-np.inf, 0.5, 1, np.inf], labels=["symmetric", "moderate", "highly skewed"])
skew

In [ ]:
plot_grid(cont_attribs, lambda c, ax: sns.histplot(df[c], kde=True, ax=ax, color=MAIN),
          title="Continuous features", name="hist_continuous")

In [ ]:
plot_grid(disc_attribs, lambda c, ax: sns.countplot(x=df[c], ax=ax, color=MAIN),
          title="Discrete features", name="count_discrete")

### 6.1.3 Binary

In [ ]:
if bin_attribs:
    binary_share = (df[bin_attribs]
                    .apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100)
                    .sort_values(ascending=False)
                    .to_frame("pct_majority_value"))
    binary_share["near_constant"] = binary_share["pct_majority_value"] > 95
    display(binary_share)
else:
    print("No binary columns.")

In [ ]:
plot_grid(bin_attribs, lambda c, ax: sns.countplot(x=df[c], ax=ax, color=MAIN),
          ncols=6, height=2.4, title="Binary features", name="count_binary")

### 6.1.4 Categorical

In [ ]:
for col in high_card:
    print(f"{col}: {df[col].nunique()} levels; top 5:")
    display(df[col].value_counts().head(5).to_frame())

In [ ]:
def plot_top_categories(col, ax):
    counts = df[col].value_counts()
    top = counts.head(TOP_N_CATEGORIES)
    sns.barplot(x=top.values, y=top.index.astype(str), ax=ax, color=MAIN, orient="h")
    ax.set_xlabel("count")
    ax.set_ylabel("")

plot_grid(cat_plot_attribs, plot_top_categories, ncols=3, height=3.5,
          title="Categorical features (top levels)", name="count_categorical")

In [ ]:
# Rare levels (< 1% of rows) - candidates for grouping into "other"
rare = {c: df[c].value_counts(normalize=True).loc[lambda s: s < 0.01].index.tolist() for c in cat_plot_attribs}
{c: v for c, v in rare.items() if v}

## 6.2 Bivariate<a id="sec-6-2"></a>

Each feature against **`recommended`**. This is where the candidate drivers come from. The charts show, for each rating value, the share of reviews that recommend; the last cell ranks the ratings by how strongly they move with `recommended`.

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    target_corr = (df[num_attribs + bin_attribs + [TARGET]]
                   .corr(method="spearman")[TARGET]
                   .drop(TARGET)
                   .sort_values(key=abs, ascending=False))
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(target_corr) + 1))
    colors = np.where(target_corr > 0, PALETTE[7], PALETTE[0])
    ax.barh(target_corr.index, target_corr.values, color=colors, height=0.7)
    ax.axvline(0, color="#52514e", linewidth=1)
    ax.invert_yaxis()
    ax.set_xlabel(f"Spearman correlation with {TARGET}")
    ax.set_title(f"Features ranked by correlation with {TARGET}")
    save_fig(fig, "target_correlation")
    plt.show()

In [ ]:
if TARGET and PROBLEM_TYPE == "regression":
    top_num = [c for c in target_corr.index if c in cont_attribs][:8]
    plot_grid(top_num,
              lambda c, ax: sns.regplot(data=df, x=c, y=TARGET, ax=ax, lowess=True,
                                        scatter_kws={"s": 8, "alpha": 0.35, "color": MAIN},
                                        line_kws={"color": PALETTE[1]}),
              title=f"Top continuous features vs {TARGET} (LOWESS trend)", name="scatter_vs_target")
elif TARGET:
    plot_grid(cont_attribs,
              lambda c, ax: sns.kdeplot(data=df, x=c, hue=TARGET, common_norm=False, fill=True,
                                        alpha=0.25, linewidth=2, ax=ax),
              title=f"Continuous features by {TARGET} class", name="dist_by_target", shared_legend=True)

In [ ]:
def plot_cat_vs_target(col, ax):
    order = df[col].value_counts().head(TOP_N_CATEGORIES).index
    if pd.api.types.is_numeric_dtype(df[col]):
        order = order.sort_values()
    if PROBLEM_TYPE == "regression":
        sns.boxplot(data=df[df[col].isin(order)], x=TARGET, y=col, order=order,
                    ax=ax, color=MAIN, fliersize=2, linewidth=1, orient="h")
    else:
        rate = pd.crosstab(df[col], df[TARGET], normalize="index").loc[order]
        rate.plot.barh(stacked=True, ax=ax, width=0.7, edgecolor="white", linewidth=1)
        ax.set_xlabel("share of rows")
    ax.set_ylabel("")

if TARGET:
    by_level = cat_plot_attribs + disc_attribs + (bin_attribs if PROBLEM_TYPE == "classification" else [])
    plot_grid(by_level, plot_cat_vs_target, ncols=3, height=3.5,
              title=f"Categorical / discrete features vs {TARGET}", name="cat_vs_target", shared_legend=True)

In [ ]:
# Binary flags: difference in median target between the two groups (a = lower value, b = higher)
if TARGET and PROBLEM_TYPE == "regression":
    rows = {}
    for col in bin_attribs:
        a, b = sorted(df[col].dropna().unique())
        y_a, y_b = df.loc[df[col] == a, TARGET].dropna(), df.loc[df[col] == b, TARGET].dropna()
        rows[col] = {"a": a, "b": b, "median_when_a": y_a.median(), "median_when_b": y_b.median(),
                     "share_of_b": (df[col] == b).mean(), "p_value": stats.mannwhitneyu(y_a, y_b).pvalue}
    lift = pd.DataFrame(rows).T
    lift["diff"] = (lift["median_when_b"] - lift["median_when_a"]).astype(float)
    display(lift.sort_values("diff", key=abs, ascending=False))

In [ ]:
# Which rating moves most with `recommended`? Spearman correlation of each rating with recommended (0/1).
# "Goes with", not "causes": this is observed data.
rating_cols = [c for c in df.select_dtypes("float").columns if c in MEANINGS]
recommended_01 = df[TARGET].astype("float")
driver_rank = (df[rating_cols].apply(lambda s: s.corr(recommended_01, method="spearman"))
               .sort_values(ascending=False).to_frame("spearman_with_recommended"))
driver_rank["n_rated"] = df[rating_cols].notna().sum()
display(driver_rank.round(2))

# Same ranking inside each of the 3 biggest segments (seat_type): does the order change?
segment_col = "seat_type"
top_segments = df[segment_col].value_counts().head(3).index
by_segment = pd.DataFrame({
    seg: df.loc[df[segment_col] == seg, rating_cols].apply(
        lambda s: s.corr(df.loc[df[segment_col] == seg, TARGET].astype("float"), method="spearman"))
    for seg in top_segments})
display(by_segment.round(2))


## 6.3 Multivariate<a id="sec-6-3"></a>

Features against **each other**: redundancy, leakage, interactions.

In [ ]:
corr_cols = num_attribs
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(0.6 * len(corr) + 3, 0.5 * len(corr) + 2))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=DIVERGING, vmin=-1, vmax=1, center=0,
                linewidths=2, linecolor="white", annot_kws={"size": 8}, cbar_kws={"shrink": 0.6}, ax=ax)
    ax.set_title("Spearman correlation (numeric features)")
    save_fig(fig, "correlation_heatmap")
    plt.show()
else:
    print("Fewer than two numeric columns: no correlation to show.")

In [ ]:
# Highly correlated pairs -> keep one of each pair (or combine) before modelling
if len(corr_cols) >= 2:
    pairs = (corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
             .stack()
             .rename("r")
             .reset_index()
             .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
    display(pairs[pairs.r.abs() >= HIGH_CORR].sort_values("r", key=abs, ascending=False))

In [ ]:
# Leakage check: ID columns or row order should NOT predict the target
if TARGET and PROBLEM_TYPE == "regression":
    for col in ID_COLS:
        if col in df.columns:
            r = df[[col, TARGET]].corr(method="spearman").iloc[0, 1]
            print(f"corr({col}, {TARGET}) = {r:.2f}" + ("  <-- rows are ordered; shuffle before splitting" if abs(r) > 0.3 else ""))

### 6.3.1 Segments

Three views that answer the business questions: (1) the mean rating and recommend rate of each segment (`HUE`), (2) the trend by review year, (3) how many airlines or airports have enough reviews to be compared.
Hue splits use at most 3 groups; smaller groups fold into "other".

In [ ]:
HUE = "seat_type"   # the column that splits the rows into segments

if HUE in df.columns:
    # Keep the 3 biggest groups, fold the rest into "other"
    segmented = df[df[HUE].notna()]
    keep = segmented[HUE].value_counts().head(3).index
    segmented = segmented.assign(**{HUE: segmented[HUE].where(segmented[HUE].isin(keep), "other")})

    rating_cols = [c for c in df.select_dtypes("float").columns if c in MEANINGS]
    mean_rating = segmented.groupby(HUE)[rating_cols].mean().T      # rows = ratings, columns = segments
    segment_size = segmented[HUE].value_counts().to_frame("n_reviews")
    segment_size["recommend_rate"] = segmented[TARGET].astype("float").groupby(segmented[HUE]).mean()
    display(segment_size.round(2))
    display(mean_rating.round(2))

    fig, ax = plt.subplots(figsize=(8, 0.5 * len(rating_cols) + 2))
    mean_rating.plot.barh(ax=ax, width=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("mean rating (1-5)")
    ax.set_title(f"Mean rating by {HUE}")
    ax.legend(title=HUE, frameon=False)
    plt.show()


In [ ]:
# Trend by review year: number of reviews, recommend rate and mean rating of each aspect
rating_cols = [c for c in df.select_dtypes("float").columns if c in MEANINGS]
yearly = df.groupby("submitted_year").agg(n_reviews=("review_id", "size"))
yearly["recommend_rate"] = df[TARGET].astype("float").groupby(df["submitted_year"]).mean()
display(yearly.round(2).join(df.groupby("submitted_year")[rating_cols].mean().round(2)))

fig, ax = plt.subplots(figsize=(8, 3))
yearly["recommend_rate"].plot(ax=ax, marker="o", color=MAIN)
ax.set_ylabel("recommend rate")
ax.set_title("Recommend rate by review year")
plt.show()


In [ ]:
# Peer rule: only compare a airline that has at least 100 reviews
reviews_per_airline = df["airline_id"].value_counts()
enough = reviews_per_airline >= 100
print(f"{enough.sum()} of {len(enough)} airlines have at least 100 reviews")
print(f"They hold {reviews_per_airline[enough].sum() / len(df):.0%} of all reviews")


In [ ]:
# Pairplot of the strongest features (slow on wide data - keep it small)
if TARGET and PROBLEM_TYPE == "regression":
    pp_cols = [c for c in target_corr.index if c in cont_attribs][:4] + [TARGET]
    g = sns.pairplot(df[pp_cols].dropna(), corner=True, plot_kws={"s": 8, "alpha": 0.35, "color": MAIN},
                     diag_kws={"color": MAIN})
    g.figure.suptitle("Pairplot: strongest continuous features", y=1.02)
    save_fig(g.figure, "pairplot")
    plt.show()

# 7. Summary of Findings<a id="sec-7"></a>

The deliverable: one answer per question from 1.2, then the actions before staging. Drafted from the data, to be reviewed.
*Numbers come from running this notebook on the CSV. Counts printed from 3.5 onward are taken after the duplicates dropped in 3.4, so they can be slightly lower than counts on the raw file.*

**Answers to the questions in 1.2**
| # | Answer |
|---|---|
| D1 | Usable after cleaning. The big issues are the messy route columns and `nationality`; nothing blocks the analysis. |
| Q1 | Lowest mean rating in every cabin is `wifi_and_connectivity` (economy 1.83, premium economy 2.25, business 2.75), then `ground_service` and `value_for_money` in economy (2.23 and 2.47). 185 of 580 airlines have at least 100 reviews and hold 94% of all reviews, so the peer rule keeps almost all the data. |
| Q2 | `value_for_money` goes with `recommended` the most (Spearman 0.83), then `ground_service` 0.77, `cabin_staff_service` 0.73, `food_and_beverages` and `seat_comfort` 0.72; `wifi_and_connectivity` and `inflight_entertainment` are lowest (0.63). The order is nearly the same in economy, premium economy and business. |
| Q3 | The recommend rate falls from 62% (2011) to 52% (2015-2016), 29% (2019) and 18% (2021-2022), then recovers to 34% (2026). Reviews peak in 2019 (16,251) and drop in 2020 (6,155). Before 2009 the rate is 0-11%, which is a recording artifact, not real opinion. |
| Q4 | Recommend rate by cabin: business 60%, premium economy 40%, economy 34%. By traveller type: see the stacked bars in 6.2 (`type_of_traveller` is 25% missing). |
| Q5 | `value_for_money` is the aspect most linked to `recommended` in every cabin (0.80 to 0.83), so spending on other aspects should be checked against it. |
| Q6 | Economy is the least satisfied group: lowest mean rating on all seven aspects and the lowest recommend rate. Nationality is only reported (747 distinct values, polluted). |
| Q7 | No pair of ratings reaches 0.8, so no rating is a copy of another. |

**Data quality**
* 156,323 rows x 26 columns; 242 rows are identical except `review_id` (156,081 after 3.4).
* `date_flown` is later than `date_submitted` in 41 rows (set to missing).
* Route: `origin = destination` in 35 rows by city and 20 by airport; a city but no airport code in 40,386 (origin), 42,571 (destination) and 8,599 (transit) rows; a 3-letter airport code typed into a city column in 4,081, 4,606 and 1,056 rows.
* `nationality`: 342 of 155,095 values do not look like a country (digits, flight numbers, very short text).
* `customer_name` ends with a number in 150,886 rows; 13,483 rows share their review text with another row, 38 of them from the same airline.
* 6 reviews are shorter than 20 characters (only a marker such as `Trip Verified`).
* Real missing data (do not fill): `transit_*` 80.7% / 86.2%, `wifi_and_connectivity` 73.6%, `aircraft` 73.6%, airport codes 51-53%, `inflight_entertainment` 41.8%, `food_and_beverages` 28.3%, `ground_service` 27.8%, `date_flown` 25.3%, `type_of_traveller` 25.1%.

**Target**
* `recommended`: 37.5% of the reviews recommend the airline. The rate depends on the period (see Q3), so compare airlines in the same period.

**Key drivers**
* Ranked by Spearman correlation with `recommended`: `value_for_money` 0.83, `ground_service` 0.77, `cabin_staff_service` 0.73, `food_and_beverages` 0.72, `seat_comfort` 0.72, `wifi_and_connectivity` 0.63, `inflight_entertainment` 0.63. Observed association, not cause.
* `value_for_money` is almost a summary of the other aspects, so it is a result to watch, not a lever to fund.

**Redundancy / leakage**
* No pair of ratings is at or above `HIGH_CORR` (0.8). `review_id` is only an identifier and is excluded.
* Reviewers fill a few more aspects over time (`n_ratings_filled` vs review year: 0.29), so missing ratings in old years are not "low scores".

**Actions before staging**
| Action | Column(s) | Why |
|---|---|---|
| Clean city text, move typed airport codes into the airport column, null junk (`stg__airline_locations`) | `origin_*`, `destination_*`, `transit_*` | messy route columns |
| Null `date_flown` when later than `date_submitted` | `date_flown` | 41 wrong dates |
| Keep only values found in the `countries` seed, null the rest | `nationality` | free text leaked in |
| Strip the trailing number | `customer_name` | not part of a real name |
| Keep one row of identical duplicates | all columns except `review_id` | 242 duplicates |
| Drop | `updated_at` | loaded in batches, no information |
| Use reviews from about 2010 on for recommend-rate comparisons | `date_submitted` | rate is near 0% before, flag not recorded |
| Do not fill nulls | `transit_*`, `aircraft`, ratings, airport columns | real missing data |


# 8. Hand-off to Modelling<a id="sec-8"></a>

Next steps (the template's modelling checklist does not apply: there is no model):

1. **Write the staging model** `stg__airline_reviews` with the cleaning actions from section 7.
2. **Add dbt tests** (`unique`, `not_null`, `relationships`, `accepted_values`) for every rule that can be tested.
3. **Run `dbt run` and `dbt test`**, then compare row counts and null counts with the numbers in this notebook.
4. **Build the star schema** (facts and dimensions) on top of the staging models.